# Case Study II: Missing values, Outliers, and Simulation Studies

- Name: Zhiyao (Ken) Yang
- Student ID: 30311195

This notebook is provided for you to use to answer questions from [Case Study 2](https://ubc-dsci.github.io/dsci-200/case-studies/case-study-2.html). 

Link to GitHub Repository: https://github.com/KEN-I-NOT/case-study-2.git

DSCI200_CaseStudy2_Zhiyao(Ken)Yang

**Milestone 1: Missing Values**

In [ ]:
library(tidyverse)
library(naniar)
library(mice)
library(simputation)
library(infer)
library(matrixStats)
library(cellWise)
library(ggplot2)

In [ ]:
library(dplyr)
set.seed(123)
storms_dat <- storms
head(storms_dat)

In [ ]:
dim(storms_dat)

The dataset contains 19537 observations and 13 variables.

In [ ]:
glimpse(storms_dat)

The dataset contains both numerical and categorical variables; numerical variables are year, month, day, hour, lat, long, wind, pressure, tropicalstorm_force_diameter, hurricane_force_diameter; categorical variables are name, status, category.

Numerical variables represent quantitative measurements, while categorical variables represent labels or groups.

In [ ]:
summary(storms_dat)

In [ ]:
colSums(is.na(storms_dat))

Category, tropicalstorm_force_diameter, hurricane_force_diameter each has at least one missing value.

Advantage: glimpse() or summary() are quick ways to get a sense of the data set so we do not have to write a lot of code to determine if there are missing data in any variable.   

Limitation: glimpse() or summary() do not show how many data are missing exactly or where in the data set missing values occur, it will be hard for us to know how the missing values are structured across all variables.

In [ ]:
library(naniar)

In [ ]:
miss_var_summary(storms_dat)

Most of the variables within this data set contain no missing information with the exception of category, tropicalstorm_force_diameter, and hurricane_force_diameter, which indicates there is some missing information for storms that did not attain a category or for wind diameters that were not reported. The majority of the core variables do not include missing values.

In [ ]:
gg_miss_var(storms_dat, show_pct = TRUE)

It is easy to visualize from this chart that indeed, most of the variables have no missing data, other than category, tropicalstorm_force_diameter, and hurricane_force_diameter. This visualization also allows us to easily find the variables with the highest proportions of missing values. However, it does not reveal new variables with missing data beyond what the summary already showed.

In [ ]:
gg_miss_fct(storms_dat, fct = status)

The plot shows that missing values vary by storm status. The variable category has a high percentage of missing values for most storm types except hurricanes. Additionally, both diameter variables show significant and identical patterns of missingness, particularly for subtropical depressions, suggesting these measurements were either not recorded or not applicable for certain storm intensities.

In [ ]:
gg_miss_upset(storms_dat)

Missing values in the data set often seem to occur together within the same observations, instead of being independent. Specifically, tropicalstorm_force_diameter and hurricane_force_diameter, as a pair, were most likely either both present or both missing for the same storms. This supports the idea that the missingness has an underlying pattern, possibly due to certain characteristics of the storms, as opposed to being randomly distributed across each variable.

In [ ]:
ggplot(storms_dat, aes(x = tropicalstorm_force_diameter, y = hurricane_force_diameter)) + geom_point()

By default, ggplot2 will exclude all cases with missing data in the columns of the data frame being plotted. This is done by default to ensure only complete data are shown in the scatter plot.

The scatter plot shows a positive relationship (difficult to determine its strength) between tropicalstorm_force_diameter and hurricane_force_diameter, meaning storms with larger tropicalstorm_force_diameter tend to also have larger hurricane_force_diameter. Many points lie at hurricane_force_diameter = 0, indicating that many storms do not reach hurricane strength even if they have tropicalstorm_force_diameters. Overall, stronger storms generally have larger wind diameters, though we can say that there is increasing variability among larger storms.

<br>
<br>

<br>
<br>

<br>

----

<br>

<br>

<br>

<br>

<br>

---- I am blanking out a bit as the graphs contain too many data points and it is taking super long to load on my laptop if I put them consecutively. ----

<br>

<br>

<br>

<br>

<br>

----

<br>
<br>
<br>

<br>
<br>

In [ ]:
ggplot(storms_dat, aes(x = tropicalstorm_force_diameter, y = hurricane_force_diameter)) +
  geom_point() +
  geom_miss_point()

Yes, the pattern of missingness is consistent with earlier observations. The missing values appear mostly where hurricane_force_diameter or tropicalstorm_force_diameter are not recorded, shown by the red points clustered where values are missing. Most observations are complete (blue points), confirming that missingness mainly occurs for the wind-diameter variables.

<br>

<br>

<br>
<br>

<br>

----

<br>

<br>
<br>
<br>

<br>

---- I am blanking out a bit as the graphs contain too many data points and it is taking super long to load on my laptop if I put them consecutively. ----

<br>

<br>

<br>

<br>
<br>

----

<br>

<br>

<br>
<br>

<br>
<br>
<br>

In [ ]:
ggplot(storms_dat, aes(x = tropicalstorm_force_diameter, y = hurricane_force_diameter, color = status)) +
  geom_point() +
  geom_miss_point()

The only status with a strong, positive correlation and a populated data for both variables is hurricane. This confirms that hurricane_force_diameter is logically exclusive to storms of that intensity. 

In [ ]:
head(storms_dat)

The data set has non-random missing values due to a relationship with storm severity and status. A majority of the core variables (ex. year, lat...) do not contain missing values, which suggests that there is consistent recording of storm information data. The majority of the missing values are found in category, tropicalstorm_force_diameter, and hurricane_force_diameter. The missing values for the category variable primarily occurred with the non-hurricane type storms, because hurricane categories apply only to hurricanes, while the missing values for the two diameter variables followed an identical and significantly missing pattern, particularly for the subtropical depression type storms. It appears therefore that the two diameter variables may be missing documentation or were considered non-applicable for ertain storm intensities. Thus we can somewhat conclude that the missingness in the dataset is not random and is related to storm intensity and status.

In [ ]:
storms_03plus <- storms_dat |>
     filter(year > 2002) |>
     mutate(tropicalstorm_force_diameter = as.numeric(tropicalstorm_force_diameter)) |>
     nabular() |>
     add_label_shadow()
head(storms_03plus)

In [ ]:
dim(storms_03plus)

In [ ]:
colSums(is.na(storms_03plus))

The category variables are only defined for storms that reach hurricane strength. For other storm statuses such as tropical depression, tropical storm, disturbance, or tropical wave, a hurricane category does not apply. Therefore, the missing values in category are structural (not applicable) rather than missing due to lack of observation. Because these values are not defined for those storms, they should not be imputed.

In [ ]:
storms_03plus <- storms_03plus |> select(-any_of("category"))

In [ ]:
head(storms_03plus)

In [ ]:
library(mice)
#mice_imp <- storms_03plus |> mice(m = 5, maxit = 10, seed = 123, printFlag = FALSE)
#comp_case <- complete(mice_imp, action = 1)

In [ ]:
library(ggplot2)

ggplot(storms_03plus, aes(x = pressure, y = tropicalstorm_force_diameter)) +
  geom_point() +
  geom_miss_point() +
  labs(x = "Pressure", y = "Tropical Storm Force Diameter")

The scatter plot shows a not strong negative relationship between pressure and tropicalstorm_force_diameter, where storms with lower pressure generally have larger tropical storm wind diameters. The missing values occur mainly for tropicalstorm_force_diameter, shown by the red points below the plot, and appear more frequently at higher pressure values. These all suggesting that this measurement is often missing for weaker storms. This indicates to us that the missingness is systematic and related to storm intensity, rather than random.

<br>
<br>
<br>
<br>
<br>

----

<br>
<br>
<br>
<br>
<br>

---- I am blanking out a bit as the graphs contain too many data points and it is taking super long to load on my laptop if I put them consecutively. ----

<br>
<br>
<br>

<br>
<br>

----

<br>
<br>
<br>
<br>
<br>
<br>
<br>

In [ ]:
ggplot(storms_03plus,
       aes(x = pressure, y = tropicalstorm_force_diameter)) +
  geom_point() +
  geom_miss_point() +
  facet_wrap(~ year) +
  labs(
    x = "Pressure",
    y = "Tropical Storm Force Diameter"
  )

This chart shows an improving trend in data quality for tropical storm force diameters between 2003 and 2022. Early years contain predominantly many missing data, while subsequent years show a dramatic increase in valid, reported measurements. For years with sufficient data, there is a clear inverse relationship: as pressure drops, the tropical storm force diameter typically increases. Overall, the data illustrates more comprehensive reporting in recent years.

In [ ]:
storms_mean <- storms_03plus |>
  mutate(tropicalstorm_force_diameter = ifelse(is.na(tropicalstorm_force_diameter), mean(tropicalstorm_force_diameter, na.rm = TRUE), tropicalstorm_force_diameter))

In [ ]:
head(storms_mean)

In [ ]:
head(storms_03plus |> select(tropicalstorm_force_diameter), n=6)

In [ ]:
head(storms_mean |> select(tropicalstorm_force_diameter), n=6)

In [ ]:
fit <- lm(tropicalstorm_force_diameter ~ pressure, data = storms_03plus)

In [ ]:
storms_lm_small <- storms_03plus |>
  mutate(tropicalstorm_force_diameter = ifelse(is.na(tropicalstorm_force_diameter), predict(fit, newdata = storms_03plus), tropicalstorm_force_diameter))

In [ ]:
head(storms_lm_small)

In [ ]:
head(storms_lm_small |> select(tropicalstorm_force_diameter), n=6)

In [ ]:
library(simputation)
storms_lm_large <- storms_03plus |> impute_lm(tropicalstorm_force_diameter ~ pressure + wind + lat + long + status)

In [ ]:
head(storms_lm_large)

In [ ]:
head(storms_lm_large |> select(tropicalstorm_force_diameter), n=6)

In [ ]:
storms_mice <- storms_03plus |> mice(m = 5, maxit = 10, seed = 123, printFlag = FALSE)
storms_mice1 <- complete(storms_mice, action = 1)

In [ ]:
bound_models <- bind_rows(
   mean = storms_mean,
   small = storms_lm_small,
   large = storms_lm_large,
   mice = storms_mice1,
   .id = "imp_model") |>
   as_tibble()
 
 bound_models |>
        arrange(tropicalstorm_force_diameter_NA) |> 
        ggplot(aes(x = pressure,
            y = tropicalstorm_force_diameter,
            colour = tropicalstorm_force_diameter_NA)) +
   geom_point() + 
   facet_wrap(~imp_model)

The mean imputation method produces a constant level for all imputed values thus creating a horizontal band and does not account for any relationship between pressure and missing values. Small regression imputation produced values that were perfectly aligned along the straight line of the linear model with respect to pressure but no variability. Large regression imputation provides more realistic imputed value spread across the data cloud since it uses multiple predictors which better reflects the structure of the data. MICE produces the most natural distribution where the imputed values are distributed within the existing data pattern therefore reflecting both uncertainty and variability in the imputations.

One limitation of mean imputation is that it replaces all missing values with the same constant value, which creates an artificial horizontal band in the plot and ignores the relationship between pressure and tropicalstorm_force_diameter. This reduces the variability in the data and can distort the distribution.



A limitation of small regression imputation is that all imputed values lie exactly on the regression line predicted from pressure. This produces a very perfectly linear pattern with no variability, making the imputed values unrealistically precise and underestimating the true uncertainty in the data by a high extend.

**Milestone 2: Imputation Assessment by Simulation**

In [ ]:
storms_complete <- storms |>
     filter(year > 2002) |>
     select(-category) |>
     drop_na()
 
# re-group storms and change variable type for later imputation

storms_pop <- storms_complete |>
   mutate(type = case_when(
     status == "hurricane" ~ "hurricane",
 
     status %in% c("tropical storm",
                   "tropical depression") ~ "tropical",
 
     status %in% c("subtropical storm",
                   "subtropical depression",
                   "tropical wave",
                   "disturbance",
                   "other low") ~ "subtropical_disturbance",
 
     status == "extratropical" ~ "extratropical"
   ),
     tropicalstorm_force_diameter = as.numeric(tropicalstorm_force_diameter))

In [ ]:
head(storms_complete)

In [ ]:
head(storms_pop)

In [ ]:
tidy(lm(tropicalstorm_force_diameter~pressure + lat + status, storms_pop))

In [ ]:
#head(storms_pop)

In [ ]:
## Generate data
get_sample <- function(dat, n_size){
  storms_sample <- dat |>
    slice_sample(n = n_size, replace = FALSE) |>
    ungroup()
  storms_sample
}

In [ ]:
## Generate missingness MCAR
generate_mcar <- function(dat, p) {
  missing_y <- sample(1:nrow(dat), size = round(p * nrow(dat)))
  dat$tropicalstorm_force_diameter[missing_y] <- NA
  dat
}

In [ ]:
## Linear regression estimation by least squares (LS)
fit_lm <- function(dat) {
  lm(tropicalstorm_force_diameter ~ pressure + lat + status, data = dat) |>
    tidy() |>
    select(term, estimate, std.error)
}

## Mean imputation, then LS estimation
est_mean_imp <- function(dat) {
  dat |>
    mutate(tropicalstorm_force_diameter =
             ifelse(is.na(tropicalstorm_force_diameter),
                    mean(tropicalstorm_force_diameter, na.rm = TRUE),
                    tropicalstorm_force_diameter)) |>
    fit_lm()
}

## Regression imputation, then LS estimation
est_lm_imp <- function(dat) {
  dat |>
    impute_lm(tropicalstorm_force_diameter ~ pressure) |>
    fit_lm()
}

## mice imputation, then LS estimation and POOLED
est_mice_imp <- function(dat, m = 5, maxit = 10, seed = 123) {
  imp <- mice(dat, m = m, maxit = maxit, seed = seed, printFlag = FALSE)
}

In [ ]:
## pooled across m imputations

imp <- mice(storms_pop, m = 5, maxit = 10, seed = 123, printFlag = FALSE)

pooled <- pool(with(imp, lm(tropicalstorm_force_diameter ~ pressure + lat + status))) |>
  summary() |>
  select(term, estimate, std.error)

pooled

In [ ]:
set.seed(123)

## Use a helper function to take a sample of size 1000
storms_sample <- get_sample(storms_pop, 1000)

## add 30% of MCAR missing values to `storms_sample`
storms_mcar <- generate_mcar(storms_sample, 0.3) |>
  nabular() |>
  add_label_shadow()

## scatter plot
ggplot(storms_mcar, aes(y = tropicalstorm_force_diameter,
                        x = pressure)) +
  geom_point(alpha = 0.4) +
  geom_miss_point() +
  theme_minimal()

Imputation and Estimation

In [ ]:
est_mice_imp <- function(dat, m = 5, maxit = 10, seed = 123) {
  imp <- mice(dat, m = m, maxit = maxit, seed = seed, printFlag = FALSE)
  pooled <- pool(with(imp, lm(tropicalstorm_force_diameter ~ pressure + lat + status))) |>
    summary() |>
    select(term, estimate, std.error)
  pooled}

In [ ]:
## mean imputation
storms_mean <- est_mean_imp(storms_mcar)

## regression imputation
storms_lm <- est_lm_imp(storms_mcar)

## mice imputation
storms_mice <- est_mice_imp(storms_mcar)

In [ ]:
cbind(fit_lm(storms_sample)[,1:2],
         complete_estimate = fit_lm(storms_mcar)$estimate,
         mean_estimate = storms_mean$estimate,
         regression_estimate = storms_lm$estimate,
         mice_estimate = storms_mice$estimate)

In [ ]:
nrow(storms_mean)
nrow(storms_lm)
nrow(storms_mice)

The results show that listwise deletion (complete case) can produce different coefficient estimates because observations with missing values are removed, reducing the sample size. Mean imputation tends to distort the relationships because it replaces missing values with a constant value, reducing variability. Regression imputation and MICE usually produce estimates closer to the original sample because they use relationships between variables to predict missing values. However, these results come from only one simulation run, so they are not sufficient to draw general conclusions about the relative performance of the methods. Multiple simulation runs would be needed to reliably compare their performances.

In [ ]:
datasets <- replicate(
  300,
  { # simulate each dataset
    simulated_data <- get_sample(storms_03plus, 1000)
    # add missing values
    generate_mcar(simulated_data, p = 0.3)
  },
  simplify = FALSE
)

complete_coeff <- datasets |>
  purrr::map_dfr(fit_lm, .id = "rep")

mean_imp_coeff <- datasets |>
  purrr::map_dfr(est_mean_imp, .id = "rep")

lm_imp_coeff <- datasets |>
  purrr::map_dfr(est_lm_imp, .id = "rep")

suppressWarnings(
  mice_imp_coeff <- datasets |>
    purrr::map_dfr(est_mice_imp, .id = "rep")
)

Note that the case study asked me to do 500 simulation runs, but 500 simulation runs would be too large for my PC to run (a saving error would occur or my PC would just crash), so I did 300 simulation runs.

In [ ]:
#run_simulations <- function() {
#  datasets <- replicate(
#    500,
#    {
#      simulated_data <- get_sample(storms_03plus, 1000)
#      generate_mcar(simulated_data, p = 0.3)
#    },
#    simplify = FALSE
#  )
#
#  complete_coeff <- datasets |> purrr::map_dfr(fit_lm, .id = "rep")
#  mean_imp_coeff <- datasets |> purrr::map_dfr(est_mean_imp, .id = "rep")
#  lm_imp_coeff   <- datasets |> purrr::map_dfr(est_lm_imp, .id = "rep")
#
#  suppressWarnings(
#    mice_imp_coeff <- datasets |> purrr::map_dfr(est_mice_imp, .id = "rep")
#  ) }
#run_simulations()

Datasets has length 300 because it is a list of 300 simulated datasets. Each element corresponds to one simulation run, where a sample of size 1000 is drawn and then 30% MCAR missingness is added. So its structure reflects the simulation design: 300 repetitions of the same data-generation process.

In [ ]:
coeffs_pressure <- bind_rows(
  complete_coeff |> mutate(method = "Complete case"),
  mean_imp_coeff |> mutate(method = "Mean imputation"),
  lm_imp_coeff |> mutate(method = "Regression imputation"),
  mice_imp_coeff |> mutate(method = "MICE imputation")
) |>
  filter(term == "pressure")
head(coeffs_pressure)

In [ ]:
dim(coeffs_pressure)

The coeffs_pressure has 1200 rows and 5 columns. This is because the simulation was repeated 300 times, and for each run the pressure coefficient was estimated using four methods: complete case, mean imputation, regression imputation, and MICE imputation. Therefore, 300×4=1200 rows, where each row represents one estimated pressure coefficient from a specific simulation run and method.

In [ ]:
storm_pop <- fit_lm(storms_pop) |>
  filter(term == "pressure") |>
  pull(estimate)

ggplot(coeffs_pressure, aes(x = estimate, fill = method)) +
  geom_histogram(
    bins = 20,
    alpha = 0.5,
    position = "identity"
  ) +
  geom_vline(xintercept = storm_pop, linetype = "dashed") +
  facet_wrap(~ method, scales = "free_y") +
  labs(
    title = "Sampling Distribution of `pressure` Coefficient Estimator",
    x = "Estimated coefficient",
    y = "Count"
  ) +
  theme(legend.position = "none")

In [ ]:
pressure_summaries <- coeffs_pressure |>
  group_by(method) |>
  summarize(
    median_beta_hat = median(estimate, na.rm = TRUE),
    median_SE = median(std.error, na.rm = TRUE),
    .groups = "drop"
  ) |>
  mutate_if(is.numeric, round, 3)

pressure_summaries

The results suggest that mean imputation produces the most biased estimate, with a median coefficient (-3.272) far from the others, and the smallest standard error, indicating underestimated variability. Complete case, regression imputation, and MICE imputation produce similar coefficient estimates (around -5.05 to -5.25), suggesting much lower biases. However, their variability differs: MICE imputation has the largest median standard error (0.367), reflecting uncertainty from multiple imputations, while regression imputation has the smallest (0.240), indicating that it may underestimate variability because imputed values lie close to the regression line.

**Milestone 3: Robust Methods in Motion Recognition**

In [ ]:
set.seed(1)

# 1) iris dataset
iris_numeric <- iris |>
  select(-Species)

# 2) Proportional to first flower, different magnitude
base <- as.numeric(iris_numeric[1, ])
factors <- seq(0.5, 2, length.out = 150)
iris_prop <- factors %*% t(base)

# 3) Similar flowers (`dir` keeps dependency among original variables)
dir <- c(1, 0.8, 1.2, 1.1)
z <- rnorm(150, sd = 0.02)

iris_similar <- iris_prop + z %*% t(dir)

# 4) Very different flowers (random, same dimensions)
iris_varied <- matrix(rnorm(150 * 4), ncol = 4)

In [ ]:
pca_prop <- prcomp(iris_prop, scale = TRUE)
pca_similar <- prcomp(iris_similar, scale = TRUE)
pca_varied <- prcomp(iris_varied, scale = TRUE)
pca_iris <- prcomp(iris_numeric, scale = TRUE)

head(pca_iris$x)

In [ ]:
ve_df <- data.frame(
  PC = rep(paste0("PC", 1:4), times = 4),
  dataset = rep(
    c("Proportional rows", "Similar rows", "True data", "Varied rows"),
    each = 4
  ),
  prop_var = c(
    (pca_prop$sdev^2) / sum(pca_prop$sdev^2),
    (pca_similar$sdev^2) / sum(pca_similar$sdev^2),
    (pca_iris$sdev^2) / sum(pca_iris$sdev^2),
    (pca_varied$sdev^2) / sum(pca_varied$sdev^2)
  )
)

ggplot(ve_df, aes(x = PC, y = prop_var, fill = dataset)) +
  geom_col(position = position_dodge(width = 0.8), width = 0.7) +
  scale_y_continuous(
    labels = scales::percent_format(accuracy = 1),
    limits = c(0, 1)
  ) +
  labs(
    x = NULL,
    y = "Proportion of variance explained",
    fill = NULL
  ) +
  theme_minimal(base_size = 12) +
  theme(
    legend.position = "top",
    panel.grid.minor = element_blank()
  )

The "Similar rows" dataset has almost all of the variances explained by PC1 since all of the measurements for each flower are nearly identical except for a tiny amount of noise around the same underlying shape. Consequently, a lot of the variations will be found in only one dimension.

The dataset that requires multiple principal components is the "Varied rows" dataset. For this dataset, the measurements have a much greater degree of independence from one another across the dimensions, therefore the variance is divided relatively equally between PC1–PC4, as opposed to being greatly concentrated into one component. Thus, several PCs would be required to summarize the structure of the data.

In [ ]:
library(cellWise)

source("case-study-2-functions.R")
data(data_dogWalker)

# save data
dogWalker_dat <- data_dogWalker
class(dogWalker_dat)

show_frame(dogWalker_dat, frame_number = 10)

In [ ]:
#show_frame(dogWalker_dat, frame_number = 20)
#show_frame(dogWalker_dat, frame_number = 30)
#show_frame(dogWalker_dat, frame_number = 40)

In [ ]:
dims <- dim(dogWalker_dat)
dims

## flatten into a dataframe
dim(dogWalker_dat) <- c(dims[1], prod(dims[2:4]))

It is quite probable that all or most of the data within the frames could be described by only a few principal components. Since the camera does not move, the background within the frames is largely consistent across frames and as such, most pixels change only slightly. Therefore, PCA should be able to account for most of the variations of the original data with a small number of principal components which are indicative of the primary visual characteristics in the scene.

The man and the dog would be considered "cell-wise" outliers as they move throughout the frames and influence only a limited number of pixels at various locations in the frame. All other pixels will continue to follow the dominant stable background patterns, and only a few pixel values will vary from the overall background patterns due to the movements of the man and the dog.

In [ ]:
## Compute PCA

In [ ]:
# The following lines add small variation where scale is zero to avoid division by 0

Xscales <- colSds(dogWalker_dat)
fix <- scale_filler_c(dogWalker_dat, Xscales, dims)

## Start with fixed data
dogWalker_dat <- fix[[1]]

dogWalker_pca <- prcomp(dogWalker_dat, scale = FALSE)

## get the loadings and visualize those of PC1
cloadings <- dogWalker_pca$rotation
cloadings_PC1 <- cloadings[, 1]

## prepare loading for image
load1_img <- prepLoading(cloadings_PC1, dims[2:4])

## visualization of the loadings of PC1
Pal <- colorRampPalette(c("blue", "white", "red"))(100)

image(t(apply(load1_img$loading, 2, rev)),
      zlim = 2*load1_img$maxab * c(-1, 1),
      col = Pal)

In this first principal component (PC1) loading image, it is easy to see the outline of the man and the dog as they move through the scene. Classical PCA has essentially created an image of the man and the dog in red and blue, which shows both where the man and the dog are at all times and the shape of them rather than just the stationary background. As such, this image supports the conclusion that the classical PCA can incorporate motion into the background estimates; and therefore does not support the hypothesis that PCA can reproduce a background without the man and the dog because their movements are still influencing the estimated background structures.

In [ ]:
locScaleX <- estLocScale(dogWalker_dat, precScale = 1e-12)

# add minimum variation to the data if scale is close to 0 and re-estimate scale
fix <- scale_filler_r(dogWalker_dat, locScaleX$scale, dims)

data_filled <- fix[[1]]
scale_fix <- fix[[2]]

# wrap `data_filled` using estimate of location and reviewed scale
Xw <- wrap(data_filled, locScaleX$loc, scale_fix)$Xw

# center wrapped data
XwC <- sweep(Xw, 2, colMeans(Xw))

# robust PCA: compute PCA on centered-wrapped data
dogWalker_rpca <- prcomp(XwC, center = FALSE, scale. = FALSE)

# get the loadings of the robust PC1
rloadings <- dogWalker_rpca$rotation
rloadings_PC1 <- rloadings[, 1]

# prepare loadings of PC1 for image
rload1_img <- prepLoading(rloadings_PC1, dims[2:4])

# visualization of the loadings of robust PC1
image(t(apply(rload1_img$loading, 2, rev)),
      zlim = 2*rload1_img$maxab * c(-1, 1),
      col = Pal)

Compared with the classical PCA loadings, the robust PCA loadings clearly show less of the man's and the dog's movements. The majority of the image is very smooth and most closely related to the background (which does not change), while the areas of the man and the dog moving appears relatively less distinct. These results illustrate how robust PCA can be used to minimize the effects of the pixels that represent moving objects (outliers), producing an estimation of the background variation that is clean compared to the non-robust PCA.

In [ ]:
## Compute the fitted frames and residuals:
aveX <- colMeans(dogWalker_dat)
Xresidual <- compute_residuals(dogWalker_dat, cloadings, aveX, k = 3)

## residuals as images

locs <- colMeans(Xresidual)
scales <- colSds(Xresidual)

show_frame_residual(20, dogWalker_dat, Xresidual, locs, scales, 20)

The residual image shows the moving man and dog very clearly while most of the background looks uniform with values that are near zero. This is an indication that classical PCA was able to extract most of the static background information and isolate the movements in the residual images. Although, it should be noted that the separation is not complete due to the presence of a few residual pixels and some noises around the background. As such, classical PCA provides partial success at separating the moving objects from the background.

In [ ]:
aveXw <- colMeans(Xw)

Xresidual <- compute_residuals(Xw, rloadings, aveXw, k = 3)

locScale_res <- estLocScale(Xresidual)

show_frame_residual(20, dogWalker_dat, Xresidual, locScale_res$loc, locScale_res$scale, 200)

In conclusion, the robust PCA produces a residual image with a greater distinction between the outlier motion (man/dog) and the majority of the background as it highlights them much more clearly than the classical PCA residual image; this is due to the presence of considerably less background noises and fewer scattered residual pixels when compared to the classical PCA.